# Entity Embedding

Goals:
* Learn how to train entity embeddings.
* Learn about entity embedding applications.
* **Age as categorical feature: Demonstrate that depreciation patterns can be learned from age when treating age as categorical feature.**
* Group vehicles into clusters: luxury, premium, mainstream, and economy.
* **Usage level as a categorical feature:** Investigate whether a DNN can learn meaningful ordinal representations from discretized numerical features.

A common approach to learning representations for categorical variables is **entity embeddings** (EE), introduced by Entity Embeddings of Categorical Variables (2016). In this work, categorical values are mapped into a continuous vector space and learned jointly with the prediction task, allowing the model to place similar categories close together in the embedding space and improve generalization on sparse data.

This idea is closely related to earlier work in NLP, particularly **Efficient Estimation of Word Representations in Vector Space**, where words are embedded such that semantic similarity is captured by distance in the embedding space.

It is also conceptually closely related to representation learning methods developed in **speech processing** (Area that I am familiar with). In particular, **i-vectors** and later **x-vectors** (X-vectors: Robust DNN Embeddings for Speaker Recognition) learn fixed-dimensional embeddings that capture underlying structure (such as speaker identity) from high-dimensional inputs.

In this tutorial we use the craigslist_vehicles dataset from Kaggle. The data contains cars listed on the Craigslist listing service. Sellers can list their cars, add descriptions, mileage, photos, and set an asking price. Buyers can browse the list of available cars (online inventory) and contact the seller if they are interested in buying.

Our main goal is to learn how to train an Entity Embedding and demonstrate that, although **age is a numerical feature**, it can be treated as categorical feature to create an **Entity Embedding that learns vehicle depreciation patterns**. We also inspect other patterns learned from categorical features such as day of the week, make, and **odometer**.


ref:
* Entity Embeddings of Categorical Variables: https://arxiv.org/abs/1604.06737
* craigslist_vehicles dataset: https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data
* X-vectors: Robust DNN Embeddings for Speaker Recognition: https://www.danielpovey.com/files/2018_icassp_xvectors.pdf

## Load and prepare enviroment

```sh
python3 -m venv ~/.venvs/entity-embedding-env

source ~/.venvs/entity-embedding-env/bin/activate

pip install --upgrade pip

pip install \
tensorflow \
tensorflow-datasets \
tensorflow-recommenders \
jupyter \
pandas \
matplotlib \
plotly 

pip install tensorflow tensorflow-datasets tensorflow-recommenders

pip install importlib-resources

# NOTE: optional if you are using vs code
pip install ipykernel

# NOTE: optional if you are using vs code
python -m ipykernel install \
--user \
--name entity-embedding \
--display-name "Python (entity-embedding)"

pip install scikit-learn

```

In [ ]:
import IPython
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import os
# NOTE: tensorflow_recommenders still use keras 2 but tensorflow switch to keras 3
# setting keras 2 as keras version for tensorflow env
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_recommenders as tfrs

print("TF:", tf.__version__)
print("TFDS:", tfds.__version__)
print("TFRS:", tfrs.__version__)

## Load dataset: Used Cars Dataset 

In [ ]:
import pandas as pd

import kagglehub
from kagglehub import KaggleDatasetAdapter

_load_fom_kaggle = False

local_file_name = "craigslist_vehicles.csv"

if _load_fom_kaggle:

    craigslist_vehicles = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "mbaabuharun/craigslist-vehicles",
        local_file_name,
    )
    # caching the file locally for future use as pkl
    craigslist_vehicles.to_csv(local_file_name, index=False)
    craigslist_vehicles.to_pickle(local_file_name.replace(".csv", ".pkl"))

else: 
    !pwd
    !ls *.csv
    #craigslist_vehicles = pd.read_csv(local_file_name)
    craigslist_vehicles = pd.read_pickle(local_file_name.replace(".csv", ".pkl"))

print(f"local file name: {local_file_name}")
print(craigslist_vehicles.shape)
craigslist_vehicles.head()



In [ ]:
craigslist_vehicles.shape

craigslist_vehicles.columns.tolist()
craigslist_vehicles.sample(11)

In [ ]:
craigslist_vehicles

## Preprocessing

In [ ]:
import numpy as np
from pandas.api.types import is_datetime64_any_dtype

def requires(criteria, message: str):

    if not criteria:
        raise ValueError(message)

    
def preprocess_data(raw_data: pd.DataFrame) -> pd.DataFrame:

    cols = [
        "price",
        "year",
        "posting_date",
        "type",
        "manufacturer",
        "model",
        "fuel",
        "transmission",
        "condition",
        "drive",
        "odometer"
    ]

    preprocessed_data = raw_data[cols].copy()
    preprocessed_data = preprocessed_data.dropna()
    preprocessed_data["posting_date"] = pd.to_datetime(preprocessed_data["posting_date"], errors="coerce")

    #preprocessed_data["posting_date"] = (pd.to_datetime(preprocessed_data["posting_date"], errors="coerce").dt.date)

    preprocessed_data["year"] = preprocessed_data["year"].astype(int)
    preprocessed_data["price"] = preprocessed_data["price"].astype(int)
    preprocessed_data["odometer"] = preprocessed_data["odometer"].astype(int)
    preprocessed_data["odometer"] = pd.to_numeric(preprocessed_data["odometer"],errors="coerce")

    return preprocessed_data


def filter_data(data: pd.DataFrame) -> pd.DataFrame:

    data = data[data["price"] > 1_000]
    data = data[data["price"] < 100_000]
    data = data[data["year"] > 1990]
    data = data[data["year"] < 2024]

    data = data[(data["odometer"] > 0) & (data["odometer"] < 500000)]

    return data


def _get_age(data: pd.DataFrame) -> pd.Series:

    age = (data["posting_date"].dt.year - data["year"]).astype("int")

    return age

def _reduce_taxonomy_cardinality(data: pd.DataFrame) -> pd.DataFrame:

    # Reduce the cardinality of the "model" column by keeping only the top 25 most frequent models
    _n_top_model = 750
    top_models = data["model"].value_counts().nlargest(_n_top_model).index
    data["model"] = np.where(data["model"].isin(top_models), data["model"], "other")

    _n_top_manufacturer = 50
    top_manufacturers = data["manufacturer"].value_counts().nlargest(_n_top_manufacturer).index
    data["manufacturer"] = np.where(data["manufacturer"].isin(top_manufacturers), data["manufacturer"], "other")

    _n_top_type = 10 
    top_type = data["type"].value_counts().nlargest(_n_top_type).index
    data["type"] = np.where(data["type"].isin(top_type), data["type"], "other")

    return data

def _get_age_as_category(age: pd.Series) -> pd.Series:

    categorical_age = age.astype(int).astype(str)
    categorical_age = np.where(age > 19, "old", categorical_age)
    
    return categorical_age

def _normalize_text_columns(data: pd.DataFrame, col: str) -> pd.DataFrame:
    
    data[col] = (
        data[col]
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # removing special chars
    special_chars = [' ', '-', '/', '(', ')', '.']
    for char in special_chars:
        data[col] = data[col].str.replace(char, '', regex=False)

    return data

def _normalize_condition_column(condition: pd.Series) -> pd.Series:

    condition_map = {
        "new": "excellent",
        "likenew": "excellent",
        "excellent": "excellent",

        "good": "medium",

        "fair": "low",
        "salvage": "low"
    }

    normalized_condition = condition.map(condition_map).fillna("unknown")

    return normalized_condition

def _get_usage_level(data: pd.DataFrame) -> pd.Series:
    data_view = data[["odometer", "age"]].copy()

    data_view["age_safe"] = data_view["age"].clip(lower=-1)
    data_view["odometer_per_age_buckets"] = data_view["odometer"] / (data_view["age_safe"] + 2)

    _percentiles = (
        data_view.groupby("age")["odometer_per_age_buckets"]
        .quantile([0.16, 0.33, 0.50, 0.66, 0.82])
        .unstack()
        .rename(columns={0.16: "p16", 0.33: "p33", 0.50: "p50", 0.66: "p66", 0.82: "p82"})
    )

    data_view = data_view.join(_percentiles, on="age")

    def _usage_bucket(row):
        if pd.isna(row["odometer_per_age_buckets"]) or pd.isna(row["p16"]):
            return "unknown"

        v = row["odometer_per_age_buckets"]

        if v <= row["p16"]:
            return "very_low"
        elif v <= row["p33"]:
            return "low"
        elif v <= row["p50"]:
            return "medium"
        elif v <= row["p66"]:
            return "high"
        elif v <= row["p82"]:
            return "very_high"
        return "extreme"

    usage_level = data_view.apply(_usage_bucket, axis=1)

    return usage_level.fillna("unknown")

def _get_time_features(data: pd.Series) -> pd.Series:

    requires("posting_date" in data.columns, "posting_date column is required to extract time features")
    requires(
        is_datetime64_any_dtype(data["posting_date"]),
        f"posting_date column must be datetime type, got: {data['posting_date'].dtype}"
    )

    posting_date = data["posting_date"]

    data["posting_day"] = posting_date.dt.day.astype(str)

    data["day_of_week"] = posting_date.dt.weekday.astype(str)

    # NOTE: skip bc redundant with day of week and posting day
    # data["is_weekend"] = (
    #     data["day_of_week"] >= 5
    # ).astype(int)

    data["posting_month"] = posting_date.dt.month.astype(str)
    # NOTE: not useful. datset is only 2 month of data
    # data["posting_quarter"] = posting_date.dt.quarter

    # data["posting_semester"] = np.where(
    #     posting_date.dt.month <= 6,
    #     1,
    #     2
    # )
    # data["posting_year"] = posting_date.dt.year

    return data

categorical_cols = [
    "type",
    "manufacturer",
    "model",
    "fuel",
    "transmission",
    "condition",
    "drive",
    "age_category",
    "usage_level",
    "posting_month",
    "posting_day",
    "day_of_week",
    # "is_weekend"
]


numerical_cols = [
    "year",
    "odometer",
    "age",
    # "odometer_per_age_buckets"
]


def feature_engineering(preprocessed_data: pd.DataFrame) -> pd.DataFrame:

    data_engineered = preprocessed_data.copy()

    for col in ["type", "manufacturer", "model", "fuel", "transmission", "condition", "drive"]:
        data_engineered = _normalize_text_columns(data_engineered, col)

    data_engineered = _reduce_taxonomy_cardinality(data_engineered)

    data_engineered["age"] = _get_age(data_engineered)
    data_engineered["age_category"] = _get_age_as_category(data_engineered["age"])

    data_engineered["usage_level"] = _get_usage_level(data_engineered)

    data_engineered["condition"] = _normalize_condition_column(data_engineered["condition"])

    data_engineered = _get_time_features(data_engineered)

    return data_engineered


In [ ]:
preprocessed_data = preprocess_data(craigslist_vehicles)
filtered_data = filter_data(preprocessed_data)
featured_data = feature_engineering(filtered_data)

craigslist_vehicles.shape
featured_data.shape
featured_data.head(5)

## EDA

In [ ]:
# NOTE: inspecting transmission and type values
featured_data.transmission.unique()
craigslist_vehicles.transmission.unique()

craigslist_vehicles.type.unique()

craigslist_vehicles.type.fillna("unknow").value_counts(normalize=True)

featured_data.groupby("transmission")["price"].describe()
featured_data.groupby("transmission")["age"].describe()

In [ ]:
craigslist_vehicles.model.value_counts(normalize=False).head(500).tail(25)

In [ ]:
# NOTE: inspecting price values impact by time features
featured_data.groupby("posting_month")["price"].describe()
# featured_data.groupby("posting_day")["price"].describe()
featured_data.groupby("day_of_week")["price"].describe()

In [ ]:
# NOTE: inspecting price values impact by age and usage level features (LLM help with this code)
usage_order = [
    "very_low",
    "low",
    "medium",
    "high",
    "very_high",
    "extreme"
]

median_tbl = (
    featured_data.groupby(
        ["age", "usage_level"]
    )["price"]
    .median()
    .unstack()
    .reindex(columns=usage_order)
)

count_tbl = (
    featured_data.groupby(
        ["age", "usage_level"]
    )["price"]
    .size()
    .unstack()
    .reindex(columns=usage_order)
)

combined = (
    median_tbl.round(0).astype(str) +
    " (n=" + count_tbl.astype(str) + ")"
)

combined.head(11)

## Train and test split

In [ ]:
from sklearn.model_selection import train_test_split

target_col = "price"

X = featured_data.drop(columns=[target_col]).copy()
y = featured_data[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train.shape
X_test.shape

# NOTE: full dataframe help erros analyses and eda
train_data = X_train.copy()
train_data[target_col] = y_train

test_data = X_test.copy()
test_data[target_col] = y_test

## Define DNN

In this section we define the Entity Embedding layers and the entire DNN.

**NOTES:**

1.  StringLookup(): similar to LabelEncoder. Motivations:

    * **handles unknown categories automatically**
        * Unknown/un seen examples are handle by the tag [UNK] and it represents of the average of unknow behavour
    * stays inside the model
    * saves vocabulary with the model
    * avoids train/serve mismatch

1. .adapt() is similar to fit(), tranform()

1.  Mental model

```
StringLookup:
    strings -> integers

Embedding:
    integers -> vectors

"ford"
  |
  v
StringLookup
  |
  v
3
  |
  v
Embedding
  |
  v
[0.12, -0.88, 0.45, ...]
```

 


```python
cat_cols = ["manufacturer", "fuel", "transmission", "day_of_week"]

inputs = {}
encoded_cat_layers = []

for col in cat_cols:
    inp = tf.keras.Input(shape=(1,), name=col, dtype=tf.string)
    lookup = tf.keras.layers.StringLookup()
    lookup.adapt(X_train[col])

    idx = lookup(inp)

    emb = tf.keras.layers.Embedding(
        input_dim=lookup.vocabulary_size(),
        output_dim=4,   # choose per feature
        name=f"{col}_emb",
    )(idx)

    vec = tf.keras.layers.Flatten()(emb)

    inputs[col] = inp
    encoded_cat_layers.append(vec)
```

numericals

```python
cat_cols = ["manufacturer", "fuel", "transmission", "day_of_week"]

inputs = {}
encoded_cat_layers = []

for col in cat_cols:
    inp = tf.keras.Input(shape=(1,), name=col, dtype=tf.string)
    lookup = tf.keras.layers.StringLookup()
    lookup.adapt(X_train[col])

    idx = lookup(inp)

    emb = tf.keras.layers.Embedding(
        input_dim=lookup.vocabulary_size(),
        output_dim=4,   # choose per feature
        name=f"{col}_emb",
    )(idx)

    vec = tf.keras.layers.Flatten()(emb)

    inputs[col] = inp
    encoded_cat_layers.append(vec)
```

In [ ]:
import math

class EmbbedingBlock(tf.keras.Model):

    def __init__(self, categorical_feature_vocabs: dict[str, list[str]]) -> None:

        super().__init__()

        requires(
            len(categorical_feature_vocabs.keys()) > 0,
            "categorical_feature_vocabs must contain at least one feature"
        )

        self.feature_vocab = categorical_feature_vocabs
        self.features = list(categorical_feature_vocabs.keys())
        self.embeddings = {}
        self.encoders = {}

        self.build_embedding_and_encoders()

    def build_embedding_and_encoders(self):

        # NOTE: define the encoders and embeddings layers and dim for each categorical feature
        # feature_vocab = {c1: vocab, c2: vocab, ...}
        for feature in self.features:

            vocab = self.feature_vocab[feature]

            requires(
                len(vocab) > 1,
                f"vocab for feature {feature} must contain at least two values"
            )

            # encoders
            self.encoders[feature] = tf.keras.layers.StringLookup(
                vocabulary=vocab,
                mask_token=None
            )

            self.embeddings[feature] = tf.keras.layers.Embedding(
                input_dim=len(vocab) + 1,
                output_dim=min(int(math.sqrt(len(vocab))), 100),
                name=f"{feature}_embedding"
            )

    def call(self, X_cat):
        embeddings = []

        for feature in self.features:

            encoder = self.encoders[feature]
            feature_idx = encoder(X_cat[feature])
            feature_emb = self.embeddings[feature](feature_idx)
            feature_emb = tf.keras.layers.Flatten()(feature_emb)
            embeddings.append(feature_emb)

        x = tf.keras.layers.Concatenate()(embeddings)
        
        return x
    
    def get_embedding_matrix(self, feature: str):

        requires(
            feature in self.features,
            f"{feature} not found"
        )

        vocab = self.encoders[feature].get_vocabulary()

        matrix = self.embeddings[feature].get_weights()[0]


        ee_dict = {k:v for k, v in zip(vocab, matrix)}


        return ee_dict

class NumericalBlock(tf.keras.Model):
    
    def __init__(self, numerical_features: list[str]) -> None:

        super().__init__()
        requires(
            len(numerical_features) > 0,
            "numerical_features must contain at least one feature"
        )

        self.features = numerical_features

        self.numerical_inputs = {}
        self.numerical_layers = []

        self.build_numerical_inputs_layers()

    def build_numerical_inputs_layers(self):

        for c in self.features:
            input_layer = tf.keras.Input(shape=(1,), name=c, dtype=tf.float32)
            self.numerical_inputs[c] = input_layer

            self.numerical_layers.append(input_layer)

    def call(self, X_num):

        layers = [tf.cast(X_num[c], tf.float32) for c in self.features]

        return tf.keras.layers.Concatenate()(layers)

class DNNModel(tf.keras.Model):

    def __init__(self, categorical_feature_vocabs: dict[str, list[str]], numerical_features: list[str]) -> None:

        super().__init__()

        self.embedding_block = EmbbedingBlock(categorical_feature_vocabs)
        self.numerical_block = NumericalBlock(numerical_features)

        self.dense1 = tf.keras.layers.Dense(128, activation="relu")
        self.dense2 = tf.keras.layers.Dense(64, activation="relu")
        self.dense3 = tf.keras.layers.Dense(32, activation="relu")
        self.out = tf.keras.layers.Dense(1, name="price")

    def call(self, X):

        x_cat = self.embedding_block(X)

        x_num = tf.keras.layers.Concatenate()([
            tf.cast(X[c], tf.float32)
            for c in self.numerical_block.features
        ])

        x = tf.keras.layers.Concatenate()([x_cat, x_num])

        x = self.dense1(x)
        x = self.dense2(x)
        x = self.dense3(x)

        return self.out(x)

    
    def predict_embedding(self, X):

        X_ee = self.embedding_block(X)
 
        return X_ee

In [ ]:
featured_data.model.unique

In [ ]:
categorical_cols = [
    "type",
    "manufacturer",
    "model",
    "fuel",
    "transmission",
    "condition",
    "drive",
    "age_category",
    "usage_level",
    "posting_month",
    "posting_day",
    "day_of_week",
    # "is_weekend"
]


numerical_cols = [
    "year",
    "odometer",
    "age",
    # "odometer_per_age_buckets"
]

model = DNNModel(
    categorical_feature_vocabs={c: featured_data[c].unique().tolist() for c in categorical_cols},
    numerical_features=numerical_cols
)

# Train Embeddings

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae"),
    ],
)

In [ ]:
train_inputs = {
    c: X_train[c].astype("string").to_numpy().reshape(-1, 1)
    for c in categorical_cols
}

train_inputs.update({
    c: X_train[c].to_numpy(dtype="float32").reshape(-1, 1)
    for c in numerical_cols
})

test_inputs = {
    c: X_test[c].astype("string").to_numpy().reshape(-1, 1)
    for c in categorical_cols
}

test_inputs.update({
    c: X_test[c].to_numpy(dtype="float32").reshape(-1, 1)
    for c in numerical_cols
})

y_train_ = y_train.to_numpy(dtype="float32")
y_test_ = y_test.to_numpy(dtype="float32")

In [ ]:
for c in X_train.columns:
    n_missing = X_train[c].isna().sum()
    
    if n_missing > 0:
        print(f"{c}: {n_missing} missing")

In [ ]:
history = model.fit(
    x=train_inputs,
    y=y_train_,
    validation_data=(test_inputs, y_test_),
    epochs=15,
    batch_size=256,
)

In [ ]:
history.history.keys()

import plotly.graph_objects as go

def plot_learning_curve(loss, val_loss):
    
    epochs = list(range(1, len(loss) + 1))

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=loss,
            mode="lines",
            name="Train Loss"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=val_loss,
            mode="lines",
            name="Validation Loss"
        )
    )

    fig.update_layout(
        title="Learning Curve",
        xaxis_title="Epoch",
        yaxis_title="Loss",
        template="plotly_white"
    )

    fig.show()

plot_learning_curve(
    history.history["mae"],
    history.history["val_mae"]
)

# Inspecting Entity Embeddings

We start by inspecting the day_of_week entity embedding because it has only 2 dimensions (since $\lfloor\sqrt{7}\rfloor = 2$), making it the simplest to reason about.

Because the data comes from a listing service, we expect a weekend vs. weekday pattern: sellers tend to post more on weekends, and buyers tend to browse more on weekends. This behavioral difference should be reflected in the embedding — weekend days (Saturday, Sunday) should have higher dot-product similarity (**higher is the dot product higher is the similarity**) to each other than to weekdays


> PS: surprisingly Thursday is not that different from Sunday

In [ ]:
day_of_week_ee = model.embedding_block.get_embedding_matrix("day_of_week")
day_of_week_ee

In [ ]:
day_of_week_ee.keys()
day_of_week_ee['0'].shape

monday = day_of_week_ee['0']
tuesday = day_of_week_ee["1"]
wednesday = day_of_week_ee["2"]
thursday = day_of_week_ee["3"]
friday = day_of_week_ee["4"]
saturday = day_of_week_ee["5"]
sunday = day_of_week_ee["6"]

np.dot(sunday, monday)
np.dot(sunday, tuesday)
np.dot(sunday, wednesday)
np.dot(sunday, thursday)
np.dot(sunday, friday)
np.dot(sunday, saturday)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# NOTE: build with help of LLM
def plot_sunday_similarity(day_of_week_ee):
    
    day_map = {
        "0": "Monday",
        "1": "Tuesday",
        "2": "Wednesday",
        "3": "Thursday",
        "4": "Friday",
        "5": "Saturday",
        "6": "Sunday",
    }

    sunday = day_of_week_ee["6"]

    rows = []
    for key, day_name in day_map.items():
        if key == "6":
            continue

        vec = day_of_week_ee[key]
        dot_sim = np.dot(sunday, vec)

        rows.append({
            "day": day_name,
            "dot_similarity_to_sunday": dot_sim,
        })

    df = pd.DataFrame(rows)

    fig = px.bar(
        df,
        x="day",
        y="dot_similarity_to_sunday",
        title="Dot-product similarity to Sunday embedding",
        labels={
            "day": "Day of week",
            "dot_similarity_to_sunday": "Dot similarity to Sunday",
        },
    )

    fig.update_layout(
        xaxis_categoryorder="array",
        xaxis_categoryarray=[
            "Monday", "Tuesday", "Wednesday",
            "Thursday", "Friday", "Saturday"
        ]
    )

    fig.show()

    return df

sunday_sim_df = plot_sunday_similarity(day_of_week_ee)
sunday_sim_df

## Age as categorical values

Age is naturally a numerical feature, but we deliberately treat it as categorical and train an entity embedding for it. The hypothesis is that the model will learn a depreciation pattern from the price signal: a 1-year-old car should have an embedding closer to a 2-year-old car than to a 10-year-old car.

For capturing this, the similarity (cosine similarity) between the age-0 embedding and each other age embedding should decrease with age.

**In addition, we observe that the inverse of the similarity delta reflects the nonlinear depreciation pattern expected for car depreciation**.

In [ ]:
age_category_ee = model.embedding_block.get_embedding_matrix("age_category")

age_category_ee.keys()

In [ ]:
age_zero = age_category_ee["0"]

# NOTE: 4 dim vector
age_zero

In [ ]:
age_zero = age_category_ee["0"]

age_zero.shape

ages = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12", "13", "14", "15", "16", "17", "18", "19", "old"]

cos_distances = {}
theta_distances = {}
norm_distances = {}
delta_distances = {}

for a in ages:

    norm_cos_distance = np.dot(age_zero, age_category_ee[a]) / (np.linalg.norm(age_zero) * np.linalg.norm(age_category_ee[a]))
    theta = np.arccos(norm_cos_distance)
    cos_distance = np.dot(age_zero, age_category_ee[a])
    delta_distance = np.linalg.norm(age_zero - age_category_ee[a])

    print(f"age: {a}, cos distance: {cos_distance}, angle (degrees): {np.degrees(theta)}; norm cos distance: {norm_cos_distance}; delta distance: {delta_distance}")

    cos_distances[a] = cos_distance
    theta_distances[a] = theta if not np.isnan(theta) else np.pi / 2.00
    norm_distances[a] = norm_cos_distance
    delta_distances[a] = delta_distance


In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_age_vs_distance(ages_cat, distances, distance_name: str = "Cosine Distance"):

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=ages_cat,
            y=distances,
            mode="lines+markers",
            name=distance_name
        )
    )

    # Force categorical ordering
    fig.update_layout(
        title=f"{distance_name} vs Age Category",
        xaxis=dict(
            title="Age Category",
            type="category",
            categoryorder="array",
            categoryarray=ages_cat
        ),
        yaxis=dict(
            title=distance_name
        ),
        template="plotly_white"
    )

    fig.show()

In [ ]:
ages = [
    "0","1","2","3","4","5","6","7","8","9",
    "10","11","12","13","14","15","16","17",
    "18","19","old"
]

# Ensure correct order
distances = np.array([
    cos_distances[a] for a in ages
])

plot_age_vs_distance(ages, distances)

In [ ]:
ages = [
    "0","1","2","3","4","5","6","7","8","9",
    "10","11","12","13","14","15","16","17",
    "18","19","old"
]

1.00/delta_distances["0"]

# Ensure correct order
distances = np.array([
    1.00/delta_distances[a] for a in ages
])


distances[0] = 1.00

plot_age_vs_distance(ages, distances, distance_name="Delta Distance")

## Luxury vs normal vehicles clusters

Because the entity embeddings have higher dimensionality, we use **t-SNE** to project them into 2D for visualization while preserving local similarity structure.


The embedding shows a clear separation between **luxury** and **non-luxury** (mainstream/economy) vehicles. Brands such as Porsche, Mercedes-Benz, and Cadillac cluster together in a distinct region, indicating that the model has learned a consistent “luxury signal” from the price data. Mean while, the **economy and mainstream** brands (e.g., Nissan, Kia, Mitsubishi) form a separate cluster, suggesting lower price levels and similar market positioning.

**Premium brands** (e.g., BMW, Audi, Lexus) tend to lie between these two extremes, acting as a bridge between luxury and mainstream segments. This is expected, as they share characteristics of both groups.

In [ ]:
train_data['manufacturer'].unique()
print()
make_ee = model.embedding_block.get_embedding_matrix("manufacturer")
make_ee['kia'].shape

make_ee.keys()

In [ ]:
bmw_emb = make_ee["bmw"]

bmw_emb.shape
bmw_emb

In [ ]:
brand_tier = {
    "kia": "economy",
    "hyundai": "economy",
    "mitsubishi": "economy",
    "fiat": "economy",
    "saturn": "economy",

    "toyota": "mainstream",
    "honda": "mainstream",
    "nissan": "mainstream",
    "mazda": "mainstream",
    "subaru": "mainstream",
    "ford": "mainstream",
    "chevrolet": "mainstream",
    "dodge": "mainstream",
    "jeep": "mainstream",
    "ram": "mainstream",
    "gmc": "mainstream",
    "chrysler": "mainstream",
    "volkswagen": "mainstream",
    "buick": "mainstream",
    "pontiac": "mainstream",
    "mercury": "mainstream",

    "acura": "premium",
    "lexus": "premium",
    "infiniti": "premium",
    "lincoln": "premium",
    "cadillac": "premium",
    "volvo": "premium",
    "audi": "premium",
    "mini": "premium",
    "landrover": "premium",
    "rover": "premium",
    "tesla": "premium",
    "alfaromeo": "premium",

    "bmw": "luxury",
    "mercedesbenz": "luxury",
    "porsche": "luxury",
    "jaguar": "luxury",
    "astonmartin": "luxury",
    "ferrari": "luxury",
}

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA


def plot_embedding_2d_by_tier(
    embedding_dict,
    brand_tier,
    method="tsne",
    perplexity=10,
    random_state=42
):
    """
    Plot 2D visualization of embedding vectors colored by brand tier.

    Parameters
    ----------
    embedding_dict : dict[str, np.ndarray]
        Example: {"toyota": array(...), "bmw": array(...)}
    brand_tier : dict[str, str]
        Example: {"toyota": "mainstream", "bmw": "luxury"}
    method : str
        "tsne" or "pca"
    perplexity : int
        Used only for TSNE
    random_state : int
        Random seed
    """

    tier_levels = ["economy", "mainstream", "premium", "luxury"]

    labels = list(embedding_dict.keys())
    X = np.array([embedding_dict[k] for k in labels])

    if method.lower() == "tsne":
        n_samples = len(labels)
        safe_perplexity = min(perplexity, max(2, n_samples - 1))
        reducer = TSNE(
            n_components=2,
            perplexity=safe_perplexity,
            random_state=random_state,
            init="pca",
            learning_rate="auto"
        )
        X_2d = reducer.fit_transform(X)

    elif method.lower() == "pca":
        reducer = PCA(n_components=2, random_state=random_state)
        X_2d = reducer.fit_transform(X)

    else:
        raise ValueError("method must be 'tsne' or 'pca'")

    df_plot = pd.DataFrame({
        "make": labels,
        "x": X_2d[:, 0],
        "y": X_2d[:, 1],
    })

    df_plot["tier"] = df_plot["make"].map(brand_tier).fillna("unknown")
    df_plot["tier"] = pd.Categorical(
        df_plot["tier"],
        categories=tier_levels + ["unknown"],
        ordered=True
    )

    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        color="tier",
        hover_name="make",
        text="make",
        category_orders={"tier": tier_levels + ["unknown"]},
        title=f"2D Embedding Visualization by Tier ({method.upper()})"
    )

    fig.update_traces(marker=dict(size=10))
    fig.update_layout(template="plotly_white")
    fig.update_traces(textposition="top center")

    fig.show()

    return df_plot


In [ ]:
df_plot = plot_embedding_2d_by_tier(
    embedding_dict=make_ee,
    brand_tier=brand_tier,
    method="tsne",
    perplexity=8
)

## Usage level

The learned usage level embedding space exhibits a monotonic behavior: `very_low < low < medium < high < very_high < extreme`, although the model was never asked to do so. The embeddings naturally organize themselves along an almost one-dimensional latent axis. Adjacent usage levels are close together, while opposite extremes are represented by vectors pointing in nearly opposite directions.

This behavior suggests that the DNN inferred the semantic ordering of the usage level categories solely from the supervised prediction task. Rather than treating the buckets as unrelated categories, the model learned that they represent progressively increasing vehicle usage. The **monotonic increase in vector norms**, together with the nearly collinear arrangement of the embeddings, indicates that entity embeddings can recover meaningful ordinal relationships from discretized numerical features without any explicit ordering constraint.

Because the embedding vectors are nearly aligned, this also suggests that a single embedding dimension should be sufficient for this feature.

In [ ]:
usage_ee = model.embedding_block.get_embedding_matrix("usage_level")
usage_ee.keys()
usage_ee['very_high'].shape

# NOTE: expecting both to return True
np.linalg.norm(usage_ee['very_low']) > np.linalg.norm(usage_ee['low']) > np.linalg.norm(usage_ee['medium']) > np.linalg.norm(usage_ee['high']) 
np.linalg.norm(usage_ee['high'])  < np.linalg.norm(usage_ee['very_high']) < np.linalg.norm(usage_ee['extreme'])

In [ ]:
def plot_ee_vector_space(embedding_dict):
    """
    Plot a 2D embedding space as vectors from the origin.

    Parameters
    ----------
    embedding_dict : dict[str, np.ndarray]
        Example:
            {
                "very_low": np.array([0.2, -0.8]),
                "medium": np.array([1.1, 0.5]),
                ...
            }
    """

    rows = []

    for label, vector in embedding_dict.items():
        if label == "[UNK]":
            continue

        rows.append({
            "label": label,
            "x": vector[0],
            "y": vector[1],
            "norm": np.linalg.norm(vector),
            "angle_deg": np.degrees(np.arctan2(vector[1], vector[0])),
        })

    df = pd.DataFrame(rows)

    fig = px.scatter(
        df,
        x="x",
        y="y",
        text="label",
        hover_data={
            "norm": ":.3f",
            "angle_deg": ":.1f",
            "x": ":.3f",
            "y": ":.3f",
        },
        title="Entity Embedding Vector Space",
        width=700,
        height=700,
    )

    # Draw vectors from origin
    for _, row in df.iterrows():
        fig.add_annotation(
            x=row["x"],
            y=row["y"],
            ax=0,
            ay=0,
            xref="x",
            yref="y",
            axref="x",
            ayref="y",
            showarrow=True,
            arrowhead=3,
            arrowsize=1,
            arrowwidth=2,
        )

    # Labels slightly above the point
    fig.update_traces(textposition="top center", marker_size=10)

    # Keep aspect ratio 1:1
    fig.update_yaxes(scaleanchor="x", scaleratio=1)

    # Draw x/y axes through the origin
    fig.add_hline(y=0, line_dash="dash", opacity=0.5)
    fig.add_vline(x=0, line_dash="dash", opacity=0.5)

    fig.show()

    return df

In [ ]:
plot_df = plot_ee_vector_space(usage_ee)
plot_df.sort_values("norm")